
<div align="center">
    
# TITANIC SURVIVAL PREDICTION
</div>

The Titanic dataset is one of the most well-known beginner-friendly datasets in Machine Learning and Data Science. In this project, we explore passenger information such as age, gender, passenger class, fare, and embarkation details to analyze the factors that influenced survival during the Titanic disaster. The notebook covers data preprocessing, exploratory data analysis, visualization, handling missing values, feature engineering, and the implementation of different Machine Learning models to predict passenger survival.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder,OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score

from sklearn.ensemble import RandomForestClassifier,  RandomForestRegressor
from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, confusion_matrix,f1_score,precision_score, classification_report, mean_absolute_error,mean_squared_error,r2_score
 
import warnings
warnings.filterwarnings('ignore')

| Library / Module                    | Purpose                                        |
| ----------------------------------- | ---------------------------------------------- |
| `pandas`                            | Data manipulation and analysis                 |
| `numpy`                             | Numerical computations                         |
| `scipy.stats`                       | Statistical analysis functions                 |
| `matplotlib.pyplot`                 | Basic data visualization                       |
| `seaborn`                           | Advanced statistical visualizations            |
| `plotly.express`                    | Interactive visualizations                     |
| `StandardScaler`                    | Standardizes feature values                    |
| `MinMaxScaler`                      | Scales values between 0 and 1                  |
| `LabelEncoder`                      | Converts categorical labels into numbers       |
| `OneHotEncoder`                     | Converts categorical data into binary columns  |
| `SimpleImputer`                     | Handles missing values using simple strategies |
| `KNNImputer`                        | Fills missing values using KNN algorithm       |
| `IterativeImputer`                  | Predicts missing values iteratively            |
| `LinearRegression`                  | Linear regression model                        |
| `train_test_split`                  | Splits dataset into training and testing sets  |
| `GridSearchCV`                      | Hyperparameter tuning                          |
| `cross_val_score`                   | Cross-validation evaluation                    |
| `RandomForestClassifier`            | Classification model using random forests      |
| `RandomForestRegressor`             | Regression model using random forests          |
| `XGBClassifier`                     | Extreme Gradient Boosting classifier           |
| `Pipeline`                          | Automates preprocessing and modeling workflow  |
| `accuracy_score`                    | Measures classification accuracy               |
| `confusion_matrix`                  | Displays prediction performance                |
| `f1_score`                          | Harmonic mean of precision and recall          |
| `precision_score`                   | Measures precision of predictions              |
| `classification_report`             | Detailed classification metrics                |
| `mean_absolute_error`               | Measures average absolute error                |
| `mean_squared_error`                | Measures squared prediction error              |
| `r2_score`                          | Measures regression performance                |
| `warnings`                          | Handles warning messages                       |
| `warnings.filterwarnings('ignore')` | Suppresses unnecessary warnings                |


# Read Dataset

In [2]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

# Dataset Basic Information

In [ ]:
df_train.head(3)

In [ ]:
df_test.head(3)

In [ ]:
print(df_train.info())

In [ ]:
print(df_test.info())

In [ ]:
df_train.describe().T

In [ ]:
df_test.describe().T

In [ ]:
df_test.corr

In [ ]:
df_train.corr

# Explore Columns

In [ ]:
df_train.columns

In [ ]:
# count of Survived 
print(df_train['Survived'].value_counts())
print("----------Train Data----------")

* 549 didnt Survive and 342 Survived

## Percentage of people by Pclass that survived

In [ ]:
print("Percentage of Pclass = 1 who survived:", df_train["Survived"][df_train["Pclass"] == 1].value_counts(normalize = True)[1]*100)

print("Percentage of Pclass = 2 who survived:", df_train["Survived"][df_train["Pclass"] == 2].value_counts(normalize = True)[1]*100)

print("Percentage of Pclass = 3 who survived:", df_train["Survived"][df_train["Pclass"] == 3].value_counts(normalize = True)[1]*100)

## Survived grouped by pclass

In [ ]:
sns.countplot(data=df_train, x='Survived', hue='Pclass')

In [ ]:
# Count of females who survived and did not survive
female_survived_count = df_train[(df_train['Sex'] == 'female') & (df_train['Survived'] == 1)].shape[0]
female_not_survived_count = df_train[(df_train['Sex'] == 'female') & (df_train['Survived'] == 0)].shape[0]

# Count of males who survived and did not survive
male_survived_count = df_train[(df_train['Sex'] == 'male') & (df_train['Survived'] == 1)].shape[0]
male_not_survived_count = df_train[(df_train['Sex'] == 'male') & (df_train['Survived'] == 0)].shape[0]

# plot
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
labels = ['Survived', 'Not Survived']
sizes = [female_survived_count, female_not_survived_count]
plt.pie(
    sizes,
    labels=labels,
    autopct='%1.1f%%',
    colors=['pink', 'red']
)
plt.title('Females')
plt.subplot(1, 2, 2)
sizes = [male_survived_count, male_not_survived_count]
plt.pie(sizes, labels=labels, autopct='%1.1f%%')
plt.title('Males')
plt.show()

* Female survived 74.2% and male 18.9%
* Female non survived 25.8% and male 81.9%

## AGE

In [ ]:
age_bins = [0, 10, 20, 30, 40, 50, 60, 70, 80]
age_labels = ['Infants', 'Toddlers', 'Kids', 'Teens', 'Youngs', 'Middle Aged', 'Old', 'Unknown']  # Add 'Unknown' for values outside the defined bins
# Categorize ages into groups
df_train['AgeGroup'] = pd.cut(df_train['Age'], bins=age_bins, labels=age_labels)

# Count survivors in each age group
survivors_by_age_group = df_train[df_train['Survived'] == 1]['AgeGroup'].value_counts()
# test data
# Define age groups
age_bins = [0, 10, 20, 30, 40, 50, 60, 70, 80]
age_labels = ['Infants', 'Toddlers', 'Kids', 'Teens', 'Youngs', 'Middle Aged', 'Old', 'Unknown']  # Add 'Unknown' for values outside the defined bins
# Categorize ages into groups
df_test['AgeGroup'] = pd.cut(df_test['Age'], bins=age_bins, labels=age_labels)

# Count survivors in each age group
survivors_by_age_group = df_train[df_train['Survived'] == 1]['AgeGroup'].value_counts()

# Plot
plt.figure(figsize=(8, 8))
plt.pie(survivors_by_age_group, labels=survivors_by_age_group.index, autopct='%1.1f%%', startangle=140)
plt.title('Survival Based on Age Groups')
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.show()

## HEATMAP

In [ ]:
plt.figure(figsize=(12,8))

corr_matrix = df_train.select_dtypes(include='number').corr()

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5,
    linecolor='white',
    square=True,
    cbar=True,
    annot_kws={"size":10}
)

plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.show()


A Heatmap is a visualization that represents values using colors.

In Data Science, heatmaps are commonly used to:

* show correlations between variables
* identify patterns
* detect strong or weak relationships

In [ ]:
#check null values in train data by using pandas
df_train.isnull().sum().sort_values(ascending=False)

* Cabin column have 687 missing values
* Age column have 177 missing values
* Embarkd column have 2 missing values

In [ ]:
# check null values in test data by using pandas
df_test.isnull().sum().sort_values(ascending=False)

* Cabin column have 327 missing values
* Age column have 86 missing values
* Fare column have 1 missing values

In [ ]:
# percentage of null values
percentage=(df_train.isnull().sum().sort_values(ascending=False)/len(df_train))*100
percentage

In [ ]:
# percentage of null values
percentage=(df_test.isnull().sum().sort_values(ascending=False)/len(df_train))*100
percentage

#  Impute Missing Values

In [ ]:
missing_data_cols = df_train.isnull().sum()[df_train.isnull().sum() > 0].index.tolist()
print("Columns  of missing values in train data :",missing_data_cols)
df_train.drop('Cabin', axis=1, inplace=True)
categorical_cols = ["Embarked",'AgeGroup']
bool_cols = []
numeric_cols = ["Age"]
missing_data_cols = df_train.isnull().sum()[df_train.isnull().sum() > 0].index.tolist()

In [ ]:
df=df_train.copy()

In [ ]:
# Encode categorical columns
def encode_data(X):

    le = LabelEncoder()

    for col in X.columns:

        if X[col].dtype == 'object' or X[col].dtype == 'category':

            X[col] = le.fit_transform(X[col].astype(str))

    return X


# Fill missing values
def fill_missing(X, cols):

    imputer = IterativeImputer(
        estimator=RandomForestRegressor(random_state=42),
        add_indicator=True
    )

    for col in cols:

        if X[col].isnull().sum() > 0:

            X[col] = imputer.fit_transform(X[[col]])[:, 0]

    return X


# Main imputation function
def impute_data(passed_col, model_type='classification'):

    # Separate null and non-null rows
    df_null = df[df[passed_col].isnull()]

    df_not_null = df[df[passed_col].notnull()]

    # Features and target
    X = df_not_null.drop(columns=[passed_col])

    y = df_not_null[passed_col]

    # Other missing columns
    other_missing_cols = [
        col for col in missing_data_cols
        if col != passed_col
    ]

    # Encode and fill missing
    X = encode_data(X)

    X = fill_missing(X, other_missing_cols)

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # Select model
    if model_type == 'classification':

        model = RandomForestClassifier(random_state=42)

    else:

        model = RandomForestRegressor(random_state=42)

    # Train model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Evaluation
    if model_type == 'classification':

        acc = accuracy_score(y_test, y_pred)

        print(f"{passed_col} Accuracy: {acc:.2f}")

    else:

        mae = mean_absolute_error(y_test, y_pred)

        rmse = mean_squared_error(y_test, y_pred) ** 0.5

        r2 = r2_score(y_test, y_pred)

        print(f"{passed_col} MAE: {mae:.2f}")

        print(f"{passed_col} RMSE: {rmse:.2f}")

        print(f"{passed_col} R2: {r2:.2f}")

    # Prepare null rows
    X_null = df_null.drop(columns=[passed_col])

    X_null = encode_data(X_null)

    X_null = fill_missing(X_null, other_missing_cols)

    # Predict missing values
    if len(df_null) > 0:

        df_null[passed_col] = model.predict(X_null)

    # Combine data
    df_combined = pd.concat([df_not_null, df_null])

    return df_combined[passed_col]


# Impute all missing columns
for col in missing_data_cols:

    missing_percent = round(
        (df[col].isnull().sum() / len(df)) * 100,
        2
    )

    print(f"\nMissing Values in {col}: {missing_percent}%")

    # Categorical columns
    if col in categorical_cols:

        df[col] = impute_data(col, "classification")

    # Numeric columns
    elif col in numeric_cols:

        df[col] = impute_data(col, "regression")

In [ ]:
df_test.isnull().sum()

In [ ]:
df_test.drop('Cabin', axis=1, inplace=True)

In [ ]:
categorical_cols = []
bool_cols = []
numeric_cols = ["Age", "Fare"]
missing_data_cols = df_test.isnull().sum()[df_test.isnull().sum() > 0].index.tolist()

In [ ]:
def impute_categorical_missing_data(passed_col):

    df_null = df_test[df_test[passed_col].isnull()]
    df_not_null = df_test[df_test[passed_col].notnull()]

    X = df_not_null.drop(passed_col, axis=1)
    y = df_not_null[passed_col]

    other_missing_cols = [
        col for col in missing_data_cols
        if col != passed_col
    ]

    label_encoder = LabelEncoder()

    # Encode categorical columns
    for col in X.columns:

        if X[col].dtype == 'object' or X[col].dtype == 'category':

            X[col] = label_encoder.fit_transform(
                X[col].astype(str)
            )

    # Encode boolean target
    if passed_col in bool_cols:

        y = label_encoder.fit_transform(y)

    # Iterative imputer
    iterative_imputer = IterativeImputer(
        estimator=RandomForestRegressor(random_state=42),
        add_indicator=True
    )

    # Fill other missing columns
    for col in other_missing_cols:

        if X[col].isnull().sum() > 0:

            imputed_values = iterative_imputer.fit_transform(
                X[[col]]
            )

            X[col] = imputed_values[:, 0]

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # Train classifier
    rf_classifier = RandomForestClassifier(random_state=42)

    rf_classifier.fit(X_train, y_train)

    # Predict
    y_pred = rf_classifier.predict(X_test)

    acc_score = accuracy_score(y_test, y_pred)

    print(
        "The feature",
        passed_col,
        "has been imputed with",
        round(acc_score * 100, 2),
        "% accuracy\n"
    )

    # Predict missing rows
    X = df_null.drop(passed_col, axis=1)

    for col in X.columns:

        if X[col].dtype == 'object' or X[col].dtype == 'category':

            X[col] = label_encoder.fit_transform(
                X[col].astype(str)
            )

    for col in other_missing_cols:

        if X[col].isnull().sum() > 0:

            imputed_values = iterative_imputer.fit_transform(
                X[[col]]
            )

            X[col] = imputed_values[:, 0]

    if len(df_null) > 0:

        df_null[passed_col] = rf_classifier.predict(X)

        if passed_col in bool_cols:

            df_null[passed_col] = df_null[
                passed_col
            ].map({0: False, 1: True})

    # Combine
    df_combined = pd.concat([df_not_null, df_null])

    return df_combined[passed_col]


def impute_continuous_missing_data(passed_col):

    df_null = df_test[df_test[passed_col].isnull()]
    df_not_null = df_test[df_test[passed_col].notnull()]

    X = df_not_null.drop(passed_col, axis=1)
    y = df_not_null[passed_col]

    other_missing_cols = [
        col for col in missing_data_cols
        if col != passed_col
    ]

    label_encoder = LabelEncoder()

    # Encode categorical columns
    for col in X.columns:

        if X[col].dtype == 'object' or X[col].dtype == 'category':

            X[col] = label_encoder.fit_transform(
                X[col].astype(str)
            )

    # Iterative imputer
    iterative_imputer = IterativeImputer(
        estimator=RandomForestRegressor(random_state=42),
        add_indicator=True
    )

    # Fill missing columns
    for col in other_missing_cols:

        if X[col].isnull().sum() > 0:

            imputed_values = iterative_imputer.fit_transform(
                X[[col]]
            )

            X[col] = imputed_values[:, 0]

    # Split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # Train regressor
    rf_regressor = RandomForestRegressor(random_state=42)

    rf_regressor.fit(X_train, y_train)

    # Predict
    y_pred = rf_regressor.predict(X_test)

    print("MAE =", mean_absolute_error(y_test, y_pred), "\n")

    print(
        "RMSE =",
        mean_squared_error(y_test, y_pred) ** 0.5,
        "\n"
    )

    print("R2 =", r2_score(y_test, y_pred), "\n")

    # Predict missing rows
    X = df_null.drop(passed_col, axis=1)

    for col in X.columns:

        if X[col].dtype == 'object' or X[col].dtype == 'category':

            X[col] = label_encoder.fit_transform(
                X[col].astype(str)
            )

    for col in other_missing_cols:

        if X[col].isnull().sum() > 0:

            imputed_values = iterative_imputer.fit_transform(
                X[[col]]
            )

            X[col] = imputed_values[:, 0]

    if len(df_null) > 0:

        df_null[passed_col] = rf_regressor.predict(X)

    # Combine
    df_combined = pd.concat([df_not_null, df_null])

    return df_combined[passed_col]


# Impute missing values
for col in missing_data_cols:

    missing_percent = round(
        (df_test[col].isnull().sum() / len(df_test)) * 100,
        2
    )

    print(
        "Missing Values",
        col,
        ":",
        str(missing_percent) + "%"
    )

    if col in categorical_cols:

        df_test[col] = impute_categorical_missing_data(col)

    elif col in numeric_cols:

        df_test[col] = impute_continuous_missing_data(col)

# LINEAR REGRESSION

In [ ]:

X = df_train[["Pclass", "Age", "Fare"]]

# Target
y = df_train["Survived"]

# Fill missing values
X = X.fillna(X.mean())

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Model
model = LinearRegression()

# Train
model.fit(X_train, y_train)

# Predictions
predictions = model.predict(X_test)

print(predictions[:10])

In [ ]:
plt.figure(figsize=(10,8))

sns.regplot(
    data=df_train,
    x="Fare",
    y="Survived",
    logistic=True,
    scatter_kws={"alpha":0.5},
    line_kws={"color":"red"}
)

plt.xlabel("Fare", fontsize=12)

plt.ylabel("Survival", fontsize=12)

plt.grid(alpha=0.3)

plt.show()

# XGBClassifier

In [ ]:
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', XGBClassifier(random_state=42))
])

# Define the hyperparameters for grid search
params = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__learning_rate': [0.05, 0.1, 0.5]
}

# Perform grid search using the pipeline and parameters
clf = GridSearchCV(pipeline, params, cv=5)
clf.fit(X_train, y_train)

# Get the best model and its parameters
best_model = clf.best_estimator_
best_model.fit(X_train, y_train)
y_pred_best = best_model.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)
conf_matrix = confusion_matrix(y_test, y_pred_best)
f1 = f1_score(y_test, y_pred_best)
precision = precision_score(y_test, y_pred_best)

In [ ]:
print("Best XGBClassifier Model:")
print("Test Accuracy:", accuracy_best)
print("F1 Score:", f1)
print("Precision Score:", precision)
print("------------------------------------------")
print("Confusion Matrix:",conf_matrix)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, cmap='Blues', fmt='g')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

# RandomForestClassifier

In [ ]:
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define the hyperparameters for grid search
params = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [10, 20, 30]
}

# Perform grid search using the pipeline and parameters
clf = GridSearchCV(pipeline, params, cv=5)
clf.fit(X_train, y_train)

# Get the best model and its parameters
best_model = clf.best_estimator_
best_model.fit(X_train, y_train)
y_pred_best = best_model.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)
conf_matrix = confusion_matrix(y_test, y_pred_best)
f1 = f1_score(y_test, y_pred_best)
precision = precision_score(y_test, y_pred_best)

# Print the results
print("Best Random Forest Model:")
print("Test Accuracy:", accuracy_best)
print("F1 Score:", f1)
print("Precision Score:", precision)
print("------------------------------------------")
print("Confusion Matrix:",conf_matrix)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, cmap='Blues', fmt='g')
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

# Ridge Regression

In [ ]:

X = df_train[["Pclass", "Age", "Fare"]]

# Target
y = df_train["Survived"]

# Fill missing values
X = X.fillna(X.mean())

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Model
model = Ridge(alpha=1.0)

# Train
model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)

# RMSE
rmse = mean_squared_error(y_test, predictions) ** 0.5

print("RMSE:", rmse)

# you’re doing an amazing job! ✨